In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
import datetime as dt

In [2]:
print(f'Last run date: {dt.datetime.today()}')

Last run date: 2024-03-11 11:31:44.082585


### Functions

In [3]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [4]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
# output
str_dirname_output = './output'

Project: 20231010-gen-xii
Task: ad_hoc
Subtask: linear_ecnl_transformation


### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Read query

In [6]:
str_filepath = './sql/query.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('SELECT tsp.bigAccountId,\n'
 'tacc.dtmStampCreation,\n'
 'tsp.intOpenBKType,\n'
 'tsp.AmtFinanced,\n'
 'tsp.BookValue,\n'
 'tloss.fltNetChgOff\n'
 'FROM electra.riskdb.analytics.tbltempstaticpool as tsp LEFT OUTER JOIN\n'
 'electra.pfsdb.dbo.tblAccount as tacc ON tsp.bigAccountId=tacc.bigAccountId '
 'LEFT OUTER JOIN\n'
 ' \n'
 ' \n'
 '\t(SELECT * \n'
 '\t FROM electra.riskdb.accountingReports.tblAccounting_LoanCOandNA_ME \n'
 "\t WHERE MonthEndDate = '2024-02-29') as tloss\n"
 ' \n'
 'ON tsp.bigAccountId=tloss.bigAccountId')


### Write into df

In [7]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# fillna
df['fltNetChgOff'].fillna(0, inplace=True)

# set to str
df['intOpenBKType'] = df['intOpenBKType'].astype(str)

# show
df

Wall time: 4.32 s


,bigAccountId,dtmStampCreation,intOpenBKType,AmtFinanced,BookValue,fltNetChgOff
0,270838,1994-10-05,nan,10290.00,8050.25,2851.77
1,270852,1994-10-14,nan,4505.44,4995.00,2350.55
2,270777,1994-10-18,nan,10343.14,8825.00,3660.45
3,270795,1994-10-26,nan,12090.32,9725.00,5853.88
4,270811,1994-10-28,nan,10731.45,9450.00,3942.34
...,...,...,...,...,...,...
413519,10130,NaT,nan,6847.67,4700.00,0.00
413520,23114,NaT,7.0,11572.92,12075.00,0.00
413521,61972,NaT,nan,9414.04,7225.00,0.00
413522,76430,NaT,nan,12976.45,10650.00,0.00


### Save

In [8]:
%%time

# save
str_filename = 'df_loss.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

Wall time: 8.27 s


### Upload to s3

In [9]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_subtask}/02_forecasted_averages/{str_filename}', 
    str_bucket_name=str_project,
)

Wall time: 2.17 s


### Clean-up

In [10]:
os.remove(str_local_path)